In [ ]:
# The previous notebook was Example_Data_Creation where we created .pkl's to save our data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [2]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
import time
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

In [3]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [4]:
# Set environment variables
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# Create new session with explicit local binding
spark = SparkSession.builder \
    .appName("EEG_Analysis") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .master("local[*]") \
    .getOrCreate()


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/07 03:23:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/07 03:23:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/07 03:23:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/04/07 03:23:10 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/04/07 03:23:10 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/04/07 03:23:10 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/04/07 03:23:10 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.


New Spark session created successfully


25/04/07 03:23:11 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context


25/04/07 03:23:25 WARN SparkContext: The path ../src/feature_extraction.py has been added already. Overwriting of added paths is not supported in the current version.
25/04/07 03:23:25 WARN SparkContext: The path ../src/preprocess_sets.py has been added already. Overwriting of added paths is not supported in the current version.
25/04/07 03:23:25 WARN SparkContext: The path ../src/schema_definition.py has been added already. Overwriting of added paths is not supported in the current version.


In [7]:
# This is how we would load the .pkl's back in (it can take a minute so be patient)
# Step 1: Load back into pandas
group_a_pandas_df_loaded = pd.read_pickle("features_alz_example.pkl")
group_c_pandas_df_loaded = pd.read_pickle("features_cntrl_example.pkl")

# Step 2: Convert to Spark DataFrames
group_a_spark_df_loaded = spark.createDataFrame(group_a_pandas_df_loaded)
group_c_spark_df_loaded = spark.createDataFrame(group_c_pandas_df_loaded)

In [8]:
type(group_a_spark_df_loaded)

pyspark.sql.dataframe.DataFrame

In [9]:
#just renaming things now that we understand the types and where things are coming from
alz_df = group_a_spark_df_loaded
cntrl_df = group_c_spark_df_loaded

In [9]:
print(alz_df.select("SubjectID", "EpochID").distinct().count())
print(cntrl_df.select("SubjectID", "EpochID").distinct().count())

25/04/07 03:21:35 WARN TaskSetManager: Stage 0 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:21:37 WARN TaskSetManager: Stage 6 contains a task of very large size (6561 KiB). The maximum recommended task size is 1000 KiB.


19546
16247


In [10]:
# Now lets do dimensionality reducton by first normalizing the power and then doing PCA.
# First step is lets split the data into training/testing

In [10]:
from dimensionality_reduction import normalize_power
# *REFERENCE* df_a_norm, df_c_norm = normalize_power(result_group_a, result_group_c)

In [11]:
cntrl_df.columns

['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']

In [12]:
NUM_TEST_SUBJECTS_PER_GROUP = 2  # i know before we had three , but 2 is better 3 took out too much data.

alz_test_subjects = (
    alz_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)


cntrl_test_subjects = (
    cntrl_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)


25/04/07 03:24:17 WARN TaskSetManager: Stage 0 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:24:18 WARN TaskSetManager: Stage 3 contains a task of very large size (6561 KiB). The maximum recommended task size is 1000 KiB.


In [13]:
print(f"azl test subjects {alz_test_subjects}\ncntrl test subjects {cntrl_test_subjects}")

azl test subjects ['sub-001', 'sub-002']
cntrl test subjects ['sub-037', 'sub-038']


In [14]:
# now we put the labels on both datasets , and split it up into testing/training

In [15]:
from pyspark.sql.functions import lit

In [16]:
# making alz have the label 1 and cntrl 0 for the ml portion ahea

In [17]:
alz_df = alz_df.withColumn("label", lit(1))
cntrl_df = cntrl_df.withColumn("label", lit(0))

In [18]:
# Filter test rows
alz_test_df = alz_df.filter(alz_df.SubjectID.isin(alz_test_subjects))
cntrl_test_df = cntrl_df.filter(cntrl_df.SubjectID.isin(cntrl_test_subjects))

# Filter training rows (not in test subjects)
alz_train_df = alz_df.filter(~alz_df.SubjectID.isin(alz_test_subjects))
cntrl_train_df = cntrl_df.filter(~cntrl_df.SubjectID.isin(cntrl_test_subjects))


In [19]:
# Individual counts
alz_train_count = alz_train_df.select("SubjectID", "EpochID").distinct().count()
alz_test_count = alz_test_df.select("SubjectID", "EpochID").distinct().count()
cntrl_train_count = cntrl_train_df.select("SubjectID", "EpochID").distinct().count()
cntrl_test_count = cntrl_test_df.select("SubjectID", "EpochID").distinct().count()

# Combined counts
test_total_count = alz_test_df.unionByName(cntrl_test_df).select("SubjectID", "EpochID").distinct().count()
train_total_count = alz_train_df.unionByName(cntrl_train_df).select("SubjectID", "EpochID").distinct().count()

# Print them out
print(f"Alzheimer's train: {alz_train_count}")
print(f"Alzheimer's test:  {alz_test_count}")
print(f"Control train:     {cntrl_train_count}")
print(f"Control test:      {cntrl_test_count}")
print(f"Total test:        {test_total_count}")
print(f"Total train:       {train_total_count}")

25/04/07 03:24:47 WARN TaskSetManager: Stage 6 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:24:48 WARN TaskSetManager: Stage 12 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:24:54 WARN TaskSetManager: Stage 18 contains a task of very large size (6561 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:24:54 WARN TaskSetManager: Stage 24 contains a task of very large size (6561 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:24:55 WARN TaskSetManager: Stage 30 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:24:56 WARN TaskSetManager: Stage 36 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
[Stage 36:=================================>                      (12 + 8) / 20]

Alzheimer's train: 18621
Alzheimer's test:  925
Control train:     15135
Control test:      1112
Total test:        2037
Total train:       33756


In [22]:
train_df = alz_train_df.unionByName(cntrl_train_df)
test_df = alz_test_df.unionByName(cntrl_test_df)


In [23]:
from dimensionality_reduction import normalize_power # it z-scores the data
# NOTE, this uses the first parameters for mean and std for the z-score, so none of test_df's data is used to z-score
train_df, test_df = normalize_power(train_df, test_df) 

In [24]:
from dimensionality_reduction import prepare_features_for_pca
# this pivots the tables so that its better suited for PCA and ML with pyspark's libraries
print(train_df.columns)
train_df, train_features_column = prepare_features_for_pca(train_df)
test_df, test_features_column = prepare_features_for_pca(test_df)
print(train_df.columns) # as we can see after they get flatened 

['Electrode', 'WaveBand', 'SubjectID', 'EpochID', 'Power', 'label']


25/04/07 03:25:02 WARN TaskSetManager: Stage 42 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:25:04 WARN TaskSetManager: Stage 45 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:25:05 WARN TaskSetManager: Stage 48 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:25:06 WARN TaskSetManager: Stage 51 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
[Stage 51:===========================>                           (10 + 10) / 20]

['SubjectID', 'EpochID', 'label', 'T4_Beta', 'Fp2_Beta', 'P3_Alpha', 'O2_Alpha', 'T3_Delta', 'F7_Delta', 'F8_Beta', 'T4_Alpha', 'F3_Delta', 'P4_Theta', 'T3_Beta', 'F7_Theta', 'T5_Theta', 'P4_Alpha', 'T6_Alpha', 'Cz_Alpha', 'T5_Alpha', 'Cz_Delta', 'F4_Alpha', 'Pz_Beta', 'Fp1_Total', 'F4_Theta', 'O2_Beta', 'O1_Beta', 'Pz_Total', 'Pz_Alpha', 'Fp2_Alpha', 'O1_Delta', 'C3_Theta', 'Cz_Theta', 'T5_Total', 'C4_Alpha', 'Cz_Total', 'T4_Total', 'P3_Total', 'Fp1_Delta', 'Fz_Delta', 'Cz_Beta', 'Fz_Total', 'T6_Total', 'F8_Alpha', 'C4_Delta', 'F4_Total', 'O1_Alpha', 'F3_Beta', 'F4_Delta', 'Fp2_Total', 'P3_Beta', 'C4_Theta', 'Pz_Delta', 'P3_Delta', 'Fp1_Beta', 'Fp1_Alpha', 'P4_Delta', 'F7_Total', 'T5_Beta', 'O2_Total', 'F7_Alpha', 'T6_Delta', 'F4_Beta', 'F8_Delta', 'F8_Total', 'O2_Theta', 'P4_Total', 'Fz_Beta', 'C4_Beta', 'T3_Total', 'F8_Theta', 'C3_Total', 'F3_Theta', 'O2_Delta', 'T4_Delta', 'Fp1_Theta', 'Fp2_Theta', 'C3_Delta', 'F3_Alpha', 'P3_Theta', 'F7_Beta', 'P4_Beta', 'O1_Total', 'O1_Theta', 'F

In [25]:
if train_features_column != test_features_column:
    print("!! VERY UNUSUAL, NEED TO DEBUG, it means that the trainig and testing have different columns :( !!")

In [26]:
from dimensionality_reduction import fit_pca_model
#Note , this finds the features to explain the model's PCA
K_VAR_TARGET=0.95
pca_model_func, k_val = fit_pca_model(train_df, train_features_column, variance_target=K_VAR_TARGET)

25/04/07 03:25:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/04/07 03:25:10 WARN TaskSetManager: Stage 54 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:25:11 WARN TaskSetManager: Stage 57 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:25:19 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/04/07 03:25:19 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
25/04/07 03:25:19 WARN TaskSetManager: Stage 77 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:25:21 WARN TaskSetManager: Stage 80 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
                         

In [27]:
print(f"We can explian {K_VAR_TARGET} with {k_val} features. That is a lot less then {len(train_df.columns)-3} (we hope).") #-3 for subjectID , epochID and lebel

We can explian 0.95 with 18 features. That is a lot less then 95 (we hope).


In [28]:
# know that we know we can explain 95% of the variance (or what we set target to) ,
# lets make our dataframces only have those important columns
from dimensionality_reduction import apply_pca_model
train_df = apply_pca_model(train_df, train_features_column, pca_model_func, k_val)
test_df  = apply_pca_model(test_df, train_features_column, pca_model_func, k_val)

In [29]:
#after PCA we have features and label, label is 0 or 1 features has all the most important features
train_df.printSchema()

root
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = false)



In [30]:
# Now that our data is labled and reduced in size, now we can do ML 

In [32]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Get input size from PCA features

mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    layers=[k_val, 100, 2],  # input → hidden (100 units) → 2 output classes
    maxIter=1000,
    seed=42
)

mlp_model = mlp.fit(train_df)
mlp_preds = mlp_model.transform(test_df)

25/04/07 03:27:17 WARN TaskSetManager: Stage 4568 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:27:18 WARN TaskSetManager: Stage 4571 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:27:20 WARN TaskSetManager: Stage 4574 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:27:21 WARN TaskSetManager: Stage 4577 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:28:43 WARN TaskSetManager: Stage 9035 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:28:44 WARN TaskSetManager: Stage 9038 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
[Stage 9040:===============>                                       (3 + 8) / 11]

In [33]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
mlp_auc = evaluator.evaluate(mlp_preds)

print("MLP AUC:", mlp_auc)

25/04/07 03:29:45 WARN TaskSetManager: Stage 9041 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:29:47 WARN TaskSetManager: Stage 9044 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
[Stage 9044:=============================================>        (17 + 3) / 20]

MLP AUC: 0.7399421543846008


In [34]:
preds_pd = mlp_preds.select("prediction", "label").toPandas()

from sklearn.metrics import classification_report, accuracy_score

print("Neural Network accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))

25/04/07 03:29:48 WARN TaskSetManager: Stage 9066 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 03:29:49 WARN TaskSetManager: Stage 9069 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Neural Network accuracy: 0.6921944035346097
              precision    recall  f1-score   support

     Control       0.75      0.65      0.70      1112
 Alzheimer's       0.64      0.74      0.69       925

    accuracy                           0.69      2037
   macro avg       0.70      0.70      0.69      2037
weighted avg       0.70      0.69      0.69      2037



In [ ]:
 # hmm , less support then expected, what is heppening , something cutting it off, need ot check size of train